In [16]:
# 采用通用模型的特征

In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor
from utils import create_excel
kl_data = pd.read_excel('../20240607FULL.xlsx', index_col = 0)

X = kl_data.drop(['惯习面', '基面直径（um）', '基面长度（um）',
                               '基面厚度（nm）', '基面析出相体积分数%', '柱面相直径（nm）', '柱面相长度（nm）',
                               '柱面相宽度（nm）', '柱面析出相体积分数%', '析出相分布','屈服强度', '抗拉强度 （UTS）','追踪编号'],axis = 1)
print('总共有{}个特征'.format(X.shape[1]))
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
    model = RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits

# 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]

# 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
create_excel(importance_output_file)
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name='kl_279features',index=False)
    print(f'文件保存到{importance_output_file}中')

scores_file= 'scores_file.xlsx'
create_excel(scores_file)
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name='kl_279eatures', index=False)
print(f'分数文件已保存到{scores_file}文件中')



总共有280个特征
[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
57.50999999999999
27.519999999999982
43.25
41.52999999999997
4.089999999999975
57.81
11.79000000000002
124.67000000000002
31.07000000000005
124.67999999999995
7.1299999999999955
<class 'pandas.core.series.Series'>
4.569999999999993
0.07999999999998408
29.600000000000023
39.49166666666662
148.43
1.5083333333333826
27.901166666666654
29.399999999999977
109.94
34.76999999999998
5.386000000000024
<class 'pandas.core.series.Series'>
12.699999999999989
55.48000000000002
42.360000000000014
110.83999999999997
5.300000000000011
0.6133333333333439
49.58047619047619
55.886666666666656
36.00999999999999
8.860000000000014
58.71933333333334
<class 'pandas.core.series.Series'>
24.20999999999998

In [18]:
# 采用289个特征

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor
from utils import create_excel
kl_data = pd.read_excel('../20240607FULL.xlsx', index_col = 0)

X = kl_data.drop(['屈服强度', '抗拉强度 （UTS）','追踪编号'],axis = 1)
print('总共有{}个特征'.format(X.shape[1]))
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
    model = RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits

# 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]

# 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
create_excel(importance_output_file)
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name='kl_289features',index=False)
    print(f'文件保存到{importance_output_file}中')

scores_file= 'scores_file.xlsx'
create_excel(scores_file)
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name='kl_289eatures', index=False)
print(f'分数文件已保存到{scores_file}文件中')



总共有290个特征
[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
58.92999999999995
23.600000000000023
39.35000000000002
43.47000000000003
3.589999999999975
62.629999999999995
12.110000000000014
114.11000000000001
20.370000000000005
112.61000000000001
6.699999999999989
<class 'pandas.core.series.Series'>
3.5299999999999727
7.330000000000041
31.029999999999973
42.120000000000005
151.51
4.389999999999986
60.94999999999999
31.370000000000005
105.51999999999998
26.730000000000018
0.6399999999999864
<class 'pandas.core.series.Series'>
16.54000000000002
64.94
31.75999999999999
92.74000000000001
9.939999999999998
8.120000000000005
48.72000000000003
46.839999999999975
33.54000000000002
0.4600000000000364
32.420000000000016
<class 'pandas.core.series.Se

In [20]:
# import pandas as pd 
# pd.read_excel('../20240607Feature.xlsx')
# important_features=pd.read_excel('average_feature_importance_kl.xlsx',sheet_name='kl_289features')
# top_feature=important_features['Feature'].head(15).to_list()
# print(top_feature)

原始Top10特征

In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='kl_22'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(),11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
5.939999999999998
28.589999999999975
31.58000000000004
42.160000000000025
42.49000000000001
31.310000000000002
10.730000000000018
106.36000000000001
5.059999999999945
84.32
6.720000000000027
<class 'pandas.core.series.Series'>
6.680000000000007
14.740000000000009
52.120000000000005
47.79000000000002
158.60000000000002
8.079999999999984
64.76999999999998
62.589999999999975
70.07999999999998
39.44
1.2799999999999727
<class 'pandas.core.series.Series'>
12.149999999999977
75.91000000000003
32.639999999999986
88.17000000000002
10.800000000000011
11.699999999999989
50.370000000000005
37.26999999999998
4.920000000000016
1.7799999999999727
14.579999999999984
<class 'pandas.core.series.Series'>
17.

,Feature,Importance
0,MagpieData range MeltingT,0.335831
1,MagpieData maximum MeltingT,0.210581
2,Interant d electrons,0.208366
3,柱面析出相体积分数%,0.049125
4,Grain Size,0.046573
5,Calculated Grain Boundary,0.023357
6,Calculated Yield Strength,0.021764
7,柱面相长度（nm）,0.017608
8,MagpieData mean GSvolume_pa,0.016890
9,柱面相直径（nm）,0.016677


文件保存到average_feature_importance_kl.xlsx中
分数文件已保存到scores_file.xlsx文件中


Top5特征

In [22]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='kl_27'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
32.79000000000002
13.990000000000009
39.73000000000002
39.72000000000003
29.680000000000007
43.889999999999986
15.430000000000007
111.50999999999999
13.029999999999973
111.58999999999997
6.639999999999986
<class 'pandas.core.series.Series'>
8.160000000000025
9.720000000000027
45.360000000000014
43.85000000000002
152.88
4.819999999999993
55.01999999999998
45.45999999999998
70.61000000000001
37.0
0.37999999999999545
<class 'pandas.core.series.Series'>
15.730000000000018
63.94
35.47000000000003
82.05000000000001
9.910000000000025
5.360000000000014
52.20999999999998
46.19
6.2099999999999795
3.169999999999959
24.94999999999999
<class 'pandas.core.series.Series'>
20.55000000000001
33.64999999999

,Feature,Importance
0,Zr fraction,0.444569
1,Interant d electrons,0.151580
2,MagpieData range MeltingT,0.087738
3,MagpieData maximum MeltingT,0.070699
4,Mn fraction,0.057008
5,Grain Size,0.030630
6,柱面析出相体积分数%,0.027018
7,Calculated Yield Strength,0.020246
8,Calculated Grain Boundary,0.016828
9,MagpieData mean GSvolume_pa,0.011392


文件保存到average_feature_importance_kl.xlsx中
分数文件已保存到scores_file.xlsx文件中


In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='kl_32'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
37.49000000000001
13.949999999999989
44.26999999999998
38.079999999999984
24.769999999999982
41.089999999999975
13.170000000000016
111.48000000000002
17.07000000000005
110.83999999999997
7.089999999999975
<class 'pandas.core.series.Series'>
7.449999999999989
0.910000000000025
47.10000000000002
41.29000000000002
152.33999999999997
4.480000000000018
65.11000000000001
47.49000000000001
73.81
40.27999999999997
1.829999999999984
<class 'pandas.core.series.Series'>
12.850000000000023
67.14999999999998
33.01999999999998
83.75
6.170000000000016
2.490000000000009
48.52999999999997
35.110000000000014
6.910000000000025
1.9700000000000273
24.980000000000018
<class 'pandas.core.series.Series'>
13.17000

,Feature,Importance
0,Zr fraction,0.407310
1,Interant d electrons,0.140077
2,MagpieData range MeltingT,0.111444
3,MagpieData maximum MeltingT,0.082494
4,Mn fraction,0.064163
5,Grain Size,0.029666
6,柱面析出相体积分数%,0.027530
7,Calculated Yield Strength,0.017110
8,Calculated Grain Boundary,0.016955
9,柱面相长度（nm）,0.011659


文件保存到average_feature_importance_kl.xlsx中
分数文件已保存到scores_file.xlsx文件中


In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='kl_16'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
26.410000000000025
43.76999999999998
38.10000000000002
35.35000000000002
63.74000000000001
34.75999999999999
15.839999999999975
104.51999999999998
9.170000000000016
102.70999999999998
7.269999999999982
<class 'pandas.core.series.Series'>
0.6999999999999886
13.740000000000009
59.5
54.26999999999998
166.94
6.75
71.73000000000002
38.48000000000002
70.31
41.0
1.9300000000000068
<class 'pandas.core.series.Series'>
2.329999999999984
68.13
30.049999999999955
81.52999999999997
17.319999999999993
25.310000000000002
50.85000000000002
32.610000000000014
0.17000000000001592
6.210000000000036
15.269999999999982
<class 'pandas.core.series.Series'>
36.360000000000014
35.0
23.450000000000045
48.9100000000

,Feature,Importance
0,MagpieData range MeltingT,0.412264
1,MagpieData maximum MeltingT,0.320848
2,柱面析出相体积分数%,0.071633
3,Grain Size,0.055868
4,Calculated Yield Strength,0.043955
5,柱面相长度（nm）,0.023051
6,柱面相直径（nm）,0.021848
7,基面析出相体积分数%,0.016010
8,基面长度（um）,0.009613
9,惯习面,0.007601


文件保存到average_feature_importance_kl.xlsx中
分数文件已保存到scores_file.xlsx文件中


In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error,r2_score
from sklearn.ensemble import RandomForestRegressor

sheet_name='kl_18'
kl_data = pd.read_excel('../20240607FULL_final.xlsx',index_col = 0,sheet_name = sheet_name)

X = kl_data.iloc[:,:-2]
y = kl_data['抗拉强度 （UTS）']
print(list(y))
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

# 初始化随机森林回归器，最大深度为40，预测器（树）数量为100
import xgboost as xgb
# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores= []
feature_importances = np.zeros(X.shape[1])

index = 0
for train_index,test_index in skf.split(X,y_binned):
    # print(train_index,test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # 初始化 XGBoost 回归器
#     model = xgb.XGBRegressor()
    model=RandomForestRegressor(random_state = 60)

    # 训练模型
    model.fit(X_train, y_train)
    feature_importances += model.feature_importances_

    # 预测测试集
    y_pred = model.predict(X_test)
    print(type(y_test))
    # print(len(y_pred))
    # print(len(y_test))
    # print(y_test)
    for i in range(len(y_test)):
        print(np.abs(y_pred[i]-y_test.iloc[i])) # 对series格式的话采用iloc进行行索引

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test,y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
print(f"Mean Absolute Percentage Error:{np.mean(mape_scores)}")
print(f"Mean Absolute Error: {np.mean(mae_scores)}")
print(f"R^2 Score: {np.mean(r2_scores)}")

average_feature_importances = feature_importances / n_splits
# 
# # 将特征重要性排序
sorted_indices = np.argsort(average_feature_importances)[::-1]
# 
# # 输出到新的表格中
importance_output_file = 'average_feature_importance_kl.xlsx'
feature_names = X.columns[sorted_indices]
feature_importance_values = average_feature_importances[sorted_indices]
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance_values})
display(feature_importance_df)
with pd.ExcelWriter(importance_output_file,engine = 'openpyxl',mode='a',if_sheet_exists = 'replace') as writer:
    feature_importance_df.to_excel(writer, sheet_name=sheet_name,index=False)
    print(f'文件保存到{importance_output_file}中')
# 
scores_file='scores_file.xlsx'
with pd.ExcelWriter(scores_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    scores_data = {'MAE': [np.mean(mae_scores)], 'MAPE': [np.mean(mape_scores)], 'R^2': [np.mean(r2_scores)]}
    df = pd.DataFrame(scores_data)
    df.to_excel(writer, sheet_name=sheet_name, index=False)
print(f'分数文件已保存到{scores_file}文件中')



[464, 493, 487, 520, 240, 220, 340, 330, 509, 511, 538, 564, 507, 538, 456, 405, 300, 436, 446, 459, 416, 398, 409, 250, 292, 280, 340, 280, 330, 280, 329, 338, 322, 274, 224, 362, 370, 250, 290, 293, 295, 296, 405, 511, 539, 564, 390, 251, 238, 277, 303, 277, 303]
<class 'pandas.core.series.Series'>
9.54000000000002
29.689999999999998
29.529999999999973
39.670000000000016
57.860000000000014
39.02999999999997
12.470000000000027
108.88999999999999
3.4599999999999795
88.35000000000002
8.220000000000027
<class 'pandas.core.series.Series'>
6.480000000000018
17.319999999999993
49.51999999999998
44.56999999999999
158.86
4.3799999999999955
75.70999999999998
50.26999999999998
68.31
41.170000000000016
3.259999999999991
<class 'pandas.core.series.Series'>
13.139999999999986
59.68000000000001
32.41999999999996
88.47000000000003
10.180000000000007
16.779999999999973
51.56999999999999
32.56999999999999
7.319999999999993
2.6000000000000227
16.120000000000005
<class 'pandas.core.series.Series'>
22.10

,Feature,Importance
0,MagpieData range MeltingT,0.275588
1,MagpieData maximum MeltingT,0.273624
2,Interant d electrons,0.215537
3,柱面析出相体积分数%,0.059596
4,Grain Size,0.051911
5,Calculated Grain Boundary,0.026826
6,Calculated Yield Strength,0.025022
7,柱面相直径（nm）,0.021301
8,柱面相长度（nm）,0.018263
9,基面析出相体积分数%,0.008982


文件保存到average_feature_importance_kl.xlsx中
分数文件已保存到scores_file.xlsx文件中


含析出相top15特征

In [26]:
# GBoost效果不如RF